# NullVector Progress Notebook

Phase: `major-changes-v2` / Phase F-J storage-backed runtime verification plus Spec 01 document-description artifacts.

This notebook exercises acquisition, tree build with summarization enabled, persisted document-description generation, gateway `invoke()` and `invoke_many()` text usage, batch-fatal escalation in `build_tree_batch()`, unified gateway visual-enrichment usage, and result inspection. It runs against the filesystem backend by default and switches to PostgreSQL when `NULLVECTOR_PROGRESS_POSTGRES_CONNINFO` is set and reachable.

The current pass also validates direct store-backed tree persistence for committed outputs, the gateway batch facade used by summarization, the new document-description builder, and the behavior where auth/config batch failures raise instead of silently returning only `BatchResult.failed`.


### Environment
This notebook prepares a clean artifact root, imports the current runtime surface, and then runs deterministic smoke tests against local fixture PDFs, including gateway batch invocation, summarized tree output persistence, document-description artifact generation, and `build_tree_batch()` behavior for non-fatal versus auth-fatal failures.


In [ ]:
# environment setup
from __future__ import annotations

import json
import os
import platform
import shutil
from pathlib import Path

cwd = Path.cwd()
REPO_ROOT = cwd if (cwd / "pyproject.toml").exists() else cwd.parent
ARTIFACT_ROOT = REPO_ROOT / "notebooks" / "_artifacts" / "progress-storage-integration"
if ARTIFACT_ROOT.exists():
    shutil.rmtree(ARTIFACT_ROOT)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

print(
    {
        "python": platform.python_version(),
        "repo_root": str(REPO_ROOT),
        "artifact_root": str(ARTIFACT_ROOT),
    }
)


In [ ]:
# imports
from pydantic import BaseModel

from nullvector.domain import (
    AcquisitionRequest,
    DocumentDescriptionRequest,
    TreeBuildRequest,
)
from nullvector.domain.common import GeometryCoordinateSpace
from nullvector.domain.tree import VisualEnrichmentRequest, VisualRegionReference
from nullvector.ingest import acquire_document
from nullvector.llm import (
    GatewayAuditConfig,
    GatewayAuthError,
    GatewayConfig,
    GatewayFailureCategory,
    GatewayRequest,
    GatewayService,
    LLMMessage,
    LLMRole,
    NoopProviderAdapter,
    NoopScriptedResponse,
    enrich_visual_region,
)
from nullvector.retrieval import DocumentDescriptionBuilder
from nullvector.storage import (
    PostgresStorageConfig,
    build_document_store,
    build_postgres_artifact_ref,
)
from nullvector.tree import build_tree
from nullvector.tree.service import build_tree_batch


In [ ]:
# configuration
BORN_DIGITAL_PDF = REPO_ROOT / "fixtures" / "pdfs" / "phase01" / "born_digital_with_outline.pdf"
VISUAL_PDF = REPO_ROOT / "fixtures" / "pdfs" / "phase01" / "mixed_content.pdf"

BORN_DIGITAL_RUN_ID = "progress-storage-acquisition"
BORN_DIGITAL_TREE_RUN_ID = "progress-storage-tree"
BORN_DIGITAL_DESCRIPTION_RUN_ID = "progress-storage-description"
VISUAL_RUN_ID = "progress-storage-visual-acquisition"
POSTGRES_CONNINFO = os.environ.get("NULLVECTOR_PROGRESS_POSTGRES_CONNINFO")
postgres_status = None
storage_config = None
if POSTGRES_CONNINFO:
    try:
        import psycopg

        with psycopg.connect(POSTGRES_CONNINFO):
            pass
        storage_config = PostgresStorageConfig(conninfo=POSTGRES_CONNINFO)
        postgres_status = "enabled"
    except Exception as exc:
        postgres_status = f"fallback-to-filesystem: {exc}"
        storage_config = None
else:
    postgres_status = "filesystem-only"

artifact_store = build_document_store(storage_config, default_filesystem_root=".")
description_builder = DocumentDescriptionBuilder(storage=storage_config)

class NotebookTextResponse(BaseModel):
    summary: str

def manifest_ref(*, run_type: str, run_id: str, document_id: str, artifact_root: str | None) -> str:
    if storage_config is None:
        assert artifact_root is not None
        return str(Path(artifact_root) / "manifest.json")
    return build_postgres_artifact_ref(
        run_type=run_type,
        run_id=run_id,
        document_id=document_id,
        artifact_path="manifest.json",
    )

def read_json_ref(ref: str) -> dict | list:
    if ref.startswith("pg://"):
        return artifact_store.read_json_artifact(ref)
    return json.loads(Path(ref).read_text(encoding="utf-8"))

gateway = GatewayService(
    GatewayConfig(
        default_model="notebook-noop-model",
        audit=GatewayAuditConfig(persist_root=str(ARTIFACT_ROOT / "gateway-audit")),
    ),
    provider_adapter=NoopProviderAdapter(
        {
            "notebook_text_demo": NoopScriptedResponse(
                output_json={
                    "summary": "The storage integration lets the same pipeline persist parser, tree, retrieval, and audit artifacts to filesystem refs or PostgreSQL refs."
                }
            ),
            "notebook_text_demo_batch_a": NoopScriptedResponse(
                output_json={"summary": "Batch item A completed through invoke_many."}
            ),
            "notebook_text_demo_batch_b": NoopScriptedResponse(
                output_json={"summary": "Batch item B completed through invoke_many."}
            ),
            "document_description": NoopScriptedResponse(
                output_json={
                    "description_text": "Born-digital outline-backed PDF focused on structured academic policy and organizational governance.",
                    "supporting_node_ids": [],
                }
            ),
            "summarize_leaf_node": NoopScriptedResponse(
                output_json={"summary": "leaf notebook summary", "keywords": ["leaf", "notebook"]}
            ),
            "summarize_parent_node": NoopScriptedResponse(
                output_json={"summary": "parent notebook summary", "keywords": ["parent", "notebook"]}
            ),
            "visual_region_enrichment": NoopScriptedResponse(
                output_json={
                    "insight": {
                        "summary": "The selected region is a rendered document image that should remain interpretive visual evidence rather than authoritative native text.",
                        "labels": ["document_scan", "visual_evidence"],
                        "attributes": {"source": "notebook-demo"},
                        "confidence": 0.95,
                    }
                }
            ),
        }
    ),
    storage=storage_config,
)

assert BORN_DIGITAL_PDF.exists(), BORN_DIGITAL_PDF
assert VISUAL_PDF.exists(), VISUAL_PDF
print(
    {
        "postgres_status": postgres_status,
        "storage_backend": "postgres" if storage_config is not None else "filesystem",
    }
)


In [ ]:
# execution
born_digital_manifest = acquire_document(
    AcquisitionRequest(
        source_path=str(BORN_DIGITAL_PDF),
        acquisition_run_id=BORN_DIGITAL_RUN_ID,
        artifact_root=str(ARTIFACT_ROOT / "acquisition_runs"),
    ),
    storage=storage_config,
)
born_digital_manifest_ref = manifest_ref(
    run_type="acquisition",
    run_id=born_digital_manifest.acquisition_run_id,
    document_id=born_digital_manifest.document_id,
    artifact_root=born_digital_manifest.artifact_root,
)
tree_manifest = build_tree(
    TreeBuildRequest(
        acquisition_manifest_path=born_digital_manifest_ref,
        tree_run_id=BORN_DIGITAL_TREE_RUN_ID,
        summarize=True,
    ),
    gateway=gateway,
    storage=storage_config,
)
tree_manifest_ref = manifest_ref(
    run_type="tree",
    run_id=tree_manifest.tree_run_id,
    document_id=tree_manifest.document_id,
    artifact_root=tree_manifest.artifact_root,
)

node_cards = read_json_ref(tree_manifest.node_cards_path)
node_summaries = read_json_ref(tree_manifest.node_summaries_path or "")
print(
    {
        "acquisition_document_id": born_digital_manifest.document_id,
        "acquisition_page_count": born_digital_manifest.page_count,
        "tree_committed_node_count": tree_manifest.committed_node_count,
        "tree_summary_count": len(node_summaries),
        "tree_titles": [card["title"] for card in node_cards],
    }
)


In [ ]:
# execution
description_manifest = description_builder.build(
    DocumentDescriptionRequest(
        acquisition_manifest_path=born_digital_manifest_ref,
        tree_manifest_path=tree_manifest_ref,
        description_run_id=BORN_DIGITAL_DESCRIPTION_RUN_ID,
    ),
    gateway=gateway,
)
document_description = read_json_ref(description_manifest.description_path)
print(
    {
        "description_run_id": description_manifest.description_run_id,
        "description_method": document_description["description_method"],
        "description_text": document_description["description_text"],
        "source_node_ids": document_description["source_node_ids"],
        "description_manifest_path": manifest_ref(
            run_type="document_description",
            run_id=description_manifest.description_run_id,
            document_id=description_manifest.document_id,
            artifact_root=description_manifest.artifact_root,
        ),
        "description_path": description_manifest.description_path,
    }
)


In [ ]:
# execution
text_result = gateway.invoke(
    GatewayRequest[NotebookTextResponse](
        operation_name="notebook_text_demo",
        messages=(
            LLMMessage(
                role=LLMRole.USER,
                content="Summarize the storage-integration impact in one sentence.",
            ),
        ),
        response_model=NotebookTextResponse,
        idempotency_key="notebook-text-demo",
    )
)
print(
    {
        "provider_name": text_result.provider_name,
        "operation_name": text_result.operation_name,
        "summary": text_result.output.summary,
        "audit_path": text_result.audit_path,
    }
)


In [ ]:
# execution
batch_results = gateway.invoke_many(
    (
        GatewayRequest[NotebookTextResponse](
            operation_name="notebook_text_demo_batch_a",
            messages=(LLMMessage(role=LLMRole.USER, content="Return batch result A."),),
            response_model=NotebookTextResponse,
            idempotency_key="notebook-text-demo-batch-a",
        ),
        GatewayRequest[NotebookTextResponse](
            operation_name="notebook_text_demo_batch_b",
            messages=(LLMMessage(role=LLMRole.USER, content="Return batch result B."),),
            response_model=NotebookTextResponse,
            idempotency_key="notebook-text-demo-batch-b",
        ),
    ),
    max_workers=2,
)
print(
    {
        "batch_count": len(batch_results),
        "batch_summaries": [result.output.summary for result in batch_results],
    }
)


In [ ]:
# execution
nonfatal_batch_gateway = GatewayService(
    GatewayConfig(
        default_model="notebook-noop-model",
        audit=GatewayAuditConfig(persist_root=str(ARTIFACT_ROOT / "gateway-audit-batch-nonfatal")),
    ),
    provider_adapter=NoopProviderAdapter(
        {
            "summarize_leaf_node": NoopScriptedResponse(
                output_json={"summary": "leaf batch summary", "keywords": ["leaf", "batch"]}
            ),
            "summarize_parent_node": NoopScriptedResponse(output_json={"wrong": "shape"}),
        }
    ),
    storage=storage_config,
)

nonfatal_batch_result = build_tree_batch(
    (
        TreeBuildRequest(
            acquisition_manifest_path=born_digital_manifest_ref,
            tree_run_id="progress-batch-nonfatal-success",
            summarize=False,
        ),
        TreeBuildRequest(
            acquisition_manifest_path=born_digital_manifest_ref,
            tree_run_id="progress-batch-nonfatal-failure",
            summarize=True,
        ),
    ),
    gateway=nonfatal_batch_gateway,
    storage=storage_config,
    max_workers=2,
)

auth_failure_gateway = GatewayService(
    GatewayConfig(
        default_model="notebook-noop-model",
        audit=GatewayAuditConfig(persist_root=str(ARTIFACT_ROOT / "gateway-audit-batch-auth")),
    ),
    provider_adapter=NoopProviderAdapter(
        {
            "summarize_leaf_node": NoopScriptedResponse(
                failure_category=GatewayFailureCategory.AUTH_FAILURE,
                failure_message="simulated auth failure",
                status_code=401,
            ),
            "summarize_parent_node": NoopScriptedResponse(
                failure_category=GatewayFailureCategory.AUTH_FAILURE,
                failure_message="simulated auth failure",
                status_code=401,
            ),
        }
    ),
    storage=storage_config,
)

auth_failure_name = None
try:
    build_tree_batch(
        (
            TreeBuildRequest(
                acquisition_manifest_path=born_digital_manifest_ref,
                tree_run_id="progress-batch-auth-one",
                summarize=True,
            ),
            TreeBuildRequest(
                acquisition_manifest_path=born_digital_manifest_ref,
                tree_run_id="progress-batch-auth-two",
                summarize=True,
            ),
        ),
        gateway=auth_failure_gateway,
        storage=storage_config,
        max_workers=2,
    )
except GatewayAuthError as exc:
    auth_failure_name = type(exc).__name__

print(
    {
        "nonfatal_successful": len(nonfatal_batch_result.successful),
        "nonfatal_failed": len(nonfatal_batch_result.failed),
        "nonfatal_error_types": [failure.error_type for failure in nonfatal_batch_result.failed],
        "auth_failure_error": auth_failure_name,
    }
)


In [ ]:
# execution
visual_manifest = acquire_document(
    AcquisitionRequest(
        source_path=str(VISUAL_PDF),
        acquisition_run_id=VISUAL_RUN_ID,
        artifact_root=str(ARTIFACT_ROOT / "acquisition_runs"),
    ),
    storage=storage_config,
)
visual_ledger = read_json_ref(visual_manifest.ledger_path)
selected_region_payload = None
selected_page_index = None
for page in visual_ledger["pages"]:
    for block in page["blocks"]:
        if block["block_type"] in {"visual_artifact", "unresolved_region"}:
            selected_region_payload = block
            selected_page_index = page["page_index"]
            break
    if selected_region_payload is not None:
        break

assert selected_region_payload is not None
assert selected_page_index is not None

visual_region = VisualRegionReference(
    document_id=visual_manifest.document_id,
    page_index=selected_page_index,
    region_id=selected_region_payload.get("visual_id", selected_region_payload.get("region_id")),
    bbox=selected_region_payload["bbox"],
    image_ref=selected_region_payload.get("image_ref"),
    asset_path=selected_region_payload.get("asset_path"),
    page_render_path=selected_region_payload.get("page_render_path"),
    render_dpi=selected_region_payload.get("render_dpi"),
    coordinate_space=GeometryCoordinateSpace(
        selected_region_payload.get("coordinate_space", "unrotated_page")
    ),
    node_id=None,
)
visual_attachment = enrich_visual_region(
    gateway,
    VisualEnrichmentRequest(
        request_id="notebook-visual-demo",
        region=visual_region,
        prompt="Describe the selected visual region conservatively and treat it as interpretive evidence.",
        metadata={"fixture": VISUAL_PDF.name},
    ),
)
print(
    {
        "region_id": visual_attachment.region_id,
        "provider_identity": visual_attachment.provider_identity,
        "confidence": visual_attachment.confidence,
        "summary": visual_attachment.insight.summary,
        "audit_path": visual_attachment.audit_path,
    }
)

In [ ]:
# inspect results
verification_report = read_json_ref(tree_manifest.verification_report_path)
strategy_report = read_json_ref(
    tree_manifest.strategy_execution_report_path or tree_manifest.build_report_path
)
summary = {
    "tree_manifest": {
        "artifact_root": tree_manifest.artifact_root,
        "committed_node_count": tree_manifest.committed_node_count,
        "unassigned_span_count": tree_manifest.unassigned_span_count,
        "node_summaries_path": tree_manifest.node_summaries_path,
    },
    "document_description": {
        "description_run_id": description_manifest.description_run_id,
        "description_method": document_description["description_method"],
        "description_text": document_description["description_text"],
        "source_node_ids": document_description["source_node_ids"],
        "description_path": description_manifest.description_path,
    },
    "strategy_report": {
        "attempted_strategies": strategy_report.get("attempted_strategies", []),
        "selected_strategy": strategy_report.get("selected_strategy"),
        "fallback_reasons": strategy_report.get("fallback_reasons", []),
    },
    "verification_status": verification_report["status"],
    "text_gateway_summary": text_result.output.summary,
    "batch_gateway_summaries": [result.output.summary for result in batch_results],
    "build_tree_batch_behavior": {
        "nonfatal_successful": len(nonfatal_batch_result.successful),
        "nonfatal_failed": len(nonfatal_batch_result.failed),
        "auth_failure_error": auth_failure_name,
    },
    "visual_enrichment_summary": visual_attachment.insight.summary,
}
print(json.dumps(summary, indent=2, sort_keys=True))


### Known Limitations
- The notebook exercises real acquisition and tree-build entrypoints against local fixture PDFs, so PyMuPDF and pypdf must be installed.
- Set `NULLVECTOR_PROGRESS_POSTGRES_CONNINFO` to exercise the PostgreSQL backend; if the connection is unavailable, the notebook falls back to the filesystem backend and prints that status in the configuration cell.
- The gateway path stays deterministic through the noop adapter so the notebook remains stable and lightweight, even when exercising `invoke_many()`, tree summarization, document-description generation, and the new `build_tree_batch()` fatal-error escalation path.
- `build_tree_batch()` still returns `BatchResult` for document-specific failures, but auth/config gateway failures now raise after worker completion; this notebook catches the auth case only so the smoke test can keep running.
- The document-description cell demonstrates the gateway-backed path; deterministic fallback remains covered by automated tests rather than a separate notebook branch.
- Committed tree outputs now persist directly through the active storage backend; filesystem runs emit absolute paths and PostgreSQL runs emit `pg://...` refs.
